# PantheraSCAN · Comparación exploratoria de rosetas (Colab)

**Módulo OPCIONAL de investigación, separado de PantheraID original.** Compara una fotografía de consulta con fotografías de referencia proporcionadas por un investigador. No reconoce automáticamente especies, no valida identidades y no necesita una base de datos externa.

**IMPORTANTE:** la biblioteca [WildlifeTools](https://github.com/WildlifeDatasets/wildlife-tools) tiene licencia MIT, pero **MegaDescriptor-T-224** se publica como **CC BY-NC 4.0 (solo usos no comerciales)**. Este cuaderno es para pruebas exploratorias **NO COMERCIALES**; no incorporamos esos pesos en Vercel, Render, una app comercial ni una API pública. Si deseás uso comercial, debemos conseguir licencia o pesos alternativos.

**Privacidad:** las fotografías se suben manualmente a tu sesión de Google Colab. No se suben a PantheraLAB ni se publican en GitHub. El JSON de salida contiene nombres de archivo y resultados, pero **no contiene fotografías, coordenadas, claves ni vectores biométricos**. No uses imágenes de personas.

**Recomendaciones científicas:** usá fotografías de costados comparables, nítidas y con rosetas visibles, preferentemente recortadas alrededor del animal. Dos fotografías parecidas pueden tener fondos similares sin ser el mismo jaguar. La similitud coseno **no es probabilidad de identidad**. Si no hay una referencia conocida comparable, no se puede confirmar identidad.

**Orden:** ejecutar todas las celdas. Primero fotografía de consulta, luego fotografías de referencia. Para cada imagen de referencia, Colab preguntará el nombre/código del individuo. No se necesita programar.

In [ ]:
%pip -q install "timm>=1.0.19" "pandas>=2.0" "Pillow>=10.0" "huggingface_hub>=0.28" "git+https://github.com/WildlifeDatasets/wildlife-tools.git"

In [ ]:
import io
import os
import json
import uuid
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import torch
import timm
from PIL import Image, ImageOps, UnidentifiedImageError
import torchvision.transforms as T
from google.colab import files
from IPython.display import display, Markdown
from wildlife_tools.data import ImageDataset
from wildlife_tools.features import DeepFeatures
from wildlife_tools.similarity import CosineSimilarity

CARPETA = Path('/content/pantherascan_rosetas_privado')
CARPETA.mkdir(exist_ok=True)
MAX_BYTES = 12 * 1024 * 1024
FORMATOS = {'.jpg', '.jpeg', '.png', '.webp'}
def preparar_imagen(nombre, datos):
    if Path(nombre).suffix.lower() not in FORMATOS or len(datos) > MAX_BYTES:
        raise ValueError('Solo JPG, PNG o WebP de hasta 12 MB.')
    try:
        with Image.open(io.BytesIO(datos)) as original:
            foto = ImageOps.exif_transpose(original).convert('RGB')
    except (UnidentifiedImageError, OSError) as exc:
        raise ValueError('Imagen inválida o dañada.') from exc
    if min(foto.size) < 100:
        print('AVISO: imagen de resolución muy baja; comparación poco confiable.')
    destino = CARPETA / (uuid.uuid4().hex + '.jpg')
    foto.save(destino, quality=93)
    return {'file': str(destino), 'filename': Path(nombre).name[:120], 'width': foto.width, 'height': foto.height}
print('Preparado. Los archivos quedan únicamente en la sesión de Colab.')

In [ ]:
print('PASO 1: subí UNA fotografía de consulta, idealmente un recorte lateral del jaguar.')
subidos = files.upload()
if len(subidos) != 1:
    raise ValueError('Subí exactamente una imagen de consulta y volvé a ejecutar esta celda.')
nombre, datos = next(iter(subidos.items()))
consulta = preparar_imagen(nombre, datos)
display(Image.open(consulta['file']).resize((320, 320)))
print('Consulta:', consulta['filename'], '|', consulta['width'], '×', consulta['height'])

In [ ]:
print('PASO 2: subí al menos una fotografía de REFERENCIA de jaguar previamente conocido.')
print('Podés subir varias fotografías del mismo jaguar o de distintos individuos.')
archivos = files.upload()
if not archivos:
    raise ValueError('Se necesita al menos una fotografía de referencia.')
if len(archivos) > 30:
    raise ValueError('Máximo 30 imágenes por ejecución.')
referencias = []
for nombre, datos in archivos.items():
    foto = preparar_imagen(nombre, datos)
    etiqueta = input('Nombre o código conocido para "' + foto['filename'] + '" (no inventar): ').strip()
    if not etiqueta or len(etiqueta) > 80:
        raise ValueError('Cada referencia necesita un nombre o código de 1 a 80 caracteres.')
    foto['identity'] = etiqueta
    referencias.append(foto)
print('Referencias:', len(referencias), '| individuos registrados:', len(set(r['identity'] for r in referencias)))
if len(referencias) < 2:
    print('ATENCIÓN: con solo una referencia no hay evaluación comparativa robusta.')

In [ ]:
print('PASO 3: consentimiento antes de descargar pesos con licencia CC BY-NC 4.0.')
print('Confirmá SOLO si se trata de una prueba de investigación NO COMERCIAL autorizada.')
acepta = input('Escribí ACEPTO para continuar (cualquier otra respuesta detiene): ').strip()
if acepta != 'ACEPTO':
    raise PermissionError('Prueba cancelada. No se descargó MegaDescriptor.')
MODELO = 'hf-hub:BVRA/MegaDescriptor-T-224'
dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Preparando modelo MegaDescriptor-T-224 con:', dispositivo)
modelo = timm.create_model(MODELO, num_classes=0, pretrained=True).eval()
transformacion = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225))
])
metadata = pd.DataFrame([
    {'path': consulta['file'], 'identity': 'QUERY_PENDING_REVIEW'},
    *[{'path': r['file'], 'identity': r['identity']} for r in referencias]
])
conjunto = ImageDataset(metadata, root=None, transform=transformacion)
extractor = DeepFeatures(modelo, batch_size=4, num_workers=0, device=dispositivo)
rasgos = extractor(conjunto)
matriz = CosineSimilarity()(rasgos, rasgos)
print('Características calculadas con WildlifeTools. Similitud coseno, NO probabilidad.')

In [ ]:
from collections import defaultdict
candidatos = defaultdict(list)
for i, ref in enumerate(referencias, start=1):
    candidatos[ref['identity']].append({
        'reference_filename': ref['filename'],
        'similarity_cosine': float(matriz[0, i])
    })
ranking = []
for etiqueta, coincidencias in candidatos.items():
    mejor = max(coincidencias, key=lambda a: a['similarity_cosine'])
    ranking.append({
        'label': etiqueta, 'best_cosine_similarity': round(mejor['similarity_cosine'], 6),
        'best_reference_filename': mejor['reference_filename'],
        'reference_images': len(coincidencias)
    })
ranking.sort(key=lambda a:a['best_cosine_similarity'], reverse=True)
print('RESULTADO: sugerencias NO verificadas; no son probabilidades.')
display(pd.DataFrame(ranking))
if len(ranking) == 1:
    print('AVISO: hay un solo individuo registrado; NO se puede decidir si la consulta corresponde a otro.')
print('Este modelo no decide si el animal es nuevo. Solo ordena similitudes con las referencias aportadas.')

In [ ]:
informe = {
  'platform':'PantheraLAB',
  'module':'PantheraSCAN',
  'schema':'pantherascan_rosetas_v1',
  'test':'optional_jaguar_individual_comparison',
  'timestamp':datetime.now(timezone.utc).isoformat(),
  'simulation':False,
  'scientifically_verified':False,
  'identity_verified':False,
  'review_required':True,
  'candidate_type':'exploratory_similarity_only',
  'species_assumed_by_user':'Panthera onca',
  'not_an_automated_species_prediction':True,
  'model':{'library':'WildlifeTools','feature_extractor':'BVRA/MegaDescriptor-T-224',
           'weights_license':'CC-BY-NC-4.0','mode':'research_noncommercial',
           'metric':'cosine_similarity','similarity_is_probability':False},
  'query':{'filename':consulta['filename'], 'width':consulta['width'],'height':consulta['height']},
  'reference_image_count':len(referencias),
  'reference_identity_count':len(ranking),
  'ranked_candidates':ranking[:20],
  'result_limitations':['same viewpoint recommended','human verification required',
                        'cannot identify unseen individuals','not validated on local jaguar dataset'],
  'saved_in_pantheralive':False,
  'saved_in_cloud_by_pantherascan':False
}
ruta = Path('/content/resultado_pantherascan_rosetas.json')
ruta.write_text(json.dumps(informe, ensure_ascii=False, indent=2),encoding='utf-8')
print('Informe JSON listo: ', ruta.name, '| sin fotografías ni rasgos biométricos.')
files.download(str(ruta))

### ¿Qué hacer con el JSON?

Abrí la versión experimental actualizada de **PantheraLAB → PantheraSCAN → Comparación opcional de rosetas** e importá `resultado_pantherascan_rosetas.json`. El científico decide si registra un nombre o descarta la sugerencia.

**Nunca interpretar una similitud de 0,95 como «95 % de identidad».** No se establecen umbrales ni porcentajes de certeza sin un conjunto de validación con individuos conocidos. Este cuaderno es investigación exploratoria, no un identificador de jaguares validado ni una herramienta de uso comercial.